<a href="https://colab.research.google.com/github/Bhagyasri-28/PriceWise-AI/blob/main/PriceWiseAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Step 1: Generate Mock Used Laptop Resale Dataset
We will programmatically create a dataset to simulate real-world laptop resale values in India based on brands, specifications, usage age, and condition.

In [ ]:
import pandas as pd
import numpy as np

# Set seed for reproducibility
np.random.seed(42)

# Generate 1000 sample records
n_samples = 1000

brands = ['Dell', 'HP', 'Lenovo', 'Asus', 'Apple']
ram_sizes = [4, 8, 16, 32] # GB
storage_sizes = [256, 512, 1024] # GB
conditions = ['Excellent', 'Good', 'Fair', 'Poor']

data = {
    'Brand': np.random.choice(brands, n_samples),
    'RAM': np.random.choice(ram_sizes, n_samples, p=[0.2, 0.5, 0.25, 0.05]),
    'Storage': np.random.choice(storage_sizes, n_samples, p=[0.4, 0.5, 0.1]),
    'Age_Years': np.random.uniform(0.5, 6.0, n_samples).round(1),
    'Condition': np.random.choice(conditions, n_samples, p=[0.25, 0.45, 0.2, 0.1])
}

df = pd.DataFrame(data)

# Base price generation logic to simulate realistic target value
def calculate_resale_price(row):
    # Base brand value
    brand_base = {'Apple': 65000, 'Dell': 40000, 'HP': 38000, 'Lenovo': 35000, 'Asus': 36000}
    price = brand_base[row['Brand']]

    # RAM factor
    price += (row['RAM'] - 8) * 2000

    # Storage factor
    price += (row['Storage'] - 256) * 15

    # Depreciation due to age
    depreciation_rate = 0.15 # 15% per year
    price = price * ((1 - depreciation_rate) ** row['Age_Years'])

    # Condition modifier
    condition_mod = {'Excellent': 1.1, 'Good': 1.0, 'Fair': 0.8, 'Poor': 0.5}
    price = price * condition_mod[row['Condition']]

    # Add some random noise
    price += np.random.normal(0, 2000)

    # Keep values reasonable and round to nearest 100
    return max(int(round(price, -2)), 5000)

df['Resale_Price_INR'] = df.apply(calculate_resale_price, axis=1)

# Save dataset to CSV
df.to_csv('used_laptops_india.csv', index=False)
print("Dataset successfully created and saved to 'used_laptops_india.csv'!")

Dataset successfully created and saved to 'used_laptops_india.csv'!


### Step 2: Load and Display the Dataset
Let's load our newly created CSV dataset and explore the first few rows.

In [ ]:
# Load the data
laptop_df = pd.read_csv('used_laptops_india.csv')

# Display column info and the first 5 rows
print(laptop_df.info())
display(laptop_df.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Brand             1000 non-null   object 
 1   RAM               1000 non-null   int64  
 2   Storage           1000 non-null   int64  
 3   Age_Years         1000 non-null   float64
 4   Condition         1000 non-null   object 
 5   Resale_Price_INR  1000 non-null   int64  
dtypes: float64(1), int64(3), object(2)
memory usage: 47.0+ KB
None


,Brand,RAM,Storage,Age_Years,Condition,Resale_Price_INR
0,Asus,16,512,0.9,Good,49200
1,Apple,8,256,3.9,Excellent,36800
2,Lenovo,8,256,5.9,Excellent,15400
3,Apple,16,256,1.5,Good,61700
4,Apple,16,1024,4.9,Good,42100


### Step 3: Data Preprocessing and Model Training
We will encode categorical columns using `LabelEncoder`, split the data into training and test sets, train a `RandomForestRegressor`, and evaluate its performance.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import joblib

# Create a copy of the dataframe for training
train_df = laptop_df.copy()

# Initialize LabelEncoders for categorical features
label_encoders = {}
for column in ['Brand', 'Condition']:
    le = LabelEncoder()
    train_df[column] = le.fit_transform(train_df[column])
    label_encoders[column] = le

# Define Input Features (X) and Target (y)
X = train_df[['Brand', 'RAM', 'Storage', 'Age_Years', 'Condition']]
y = train_df['Resale_Price_INR']

# Split into 80% Training and 20% Testing data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train Random Forest Regressor Model
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Make Predictions on Test data
y_pred = model.predict(X_test)

# Evaluate the Model
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Model Training Complete!")
print(f"Mean Absolute Error (MAE): Rs. {mae:.2f}")
print(f"Model Accuracy (R² Score): {r2 * 100:.2f}%")

# Save the Model and Encoders for our Chatbot
model_data = {
    'model': model,
    'label_encoders': label_encoders
}
joblib.dump(model_data, 'price_wise_model.pkl')
print("Model saved successfully as 'price_wise_model.pkl'!")

Model Training Complete!
Mean Absolute Error (MAE): Rs. 3144.70
Model Accuracy (R² Score): 90.27%
Model saved successfully as 'price_wise_model.pkl'!


### Step 4: Let's Test a Real Prediction!
Now we can write a function to simulate user input from our chatbot and get a price prediction.

In [ ]:
def predict_laptop_price(brand, ram, storage, age, condition):
    # Load model and encoders
    saved_data = joblib.load('price_wise_model.pkl')
    saved_model = saved_data['model']
    encoders = saved_data['label_encoders']

    try:
        # Encode Brand and Condition input using our fitted encoders
        encoded_brand = encoders['Brand'].transform([brand])[0]
        encoded_condition = encoders['Condition'].transform([condition])[0]
    except ValueError as e:
        return f"Error in input values: {e}"

    # Prepare features array
    features = [[encoded_brand, ram, storage, age, encoded_condition]]

    # Predict price
    predicted_val = saved_model.predict(features)[0]

    # Generate a reasonable estimation range
    lower_bound = int(round(predicted_val * 0.9, -2))
    upper_bound = int(round(predicted_val * 1.1, -2))

    return f"💬 PriceWise AI: Mee laptop details batti estimated resale price Rs. {lower_bound:,} - Rs. {upper_bound:,} undavachu."

# Example test run
sample_prediction = predict_laptop_price(brand='Dell', ram=8, storage=512, age=2.0, condition='Good')
print(sample_prediction)

💬 PriceWise AI: Mee laptop details batti estimated resale price Rs. 28,800 - Rs. 35,200 undavachu.


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(


### Step 3: Data Preprocessing and Model Training
We will encode categorical columns using `LabelEncoder`, split the data into training and test sets, train a `RandomForestRegressor`, and evaluate its performance.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import joblib

# Create a copy of the dataframe for training
train_df = laptop_df.copy()

# Initialize LabelEncoders for categorical features
label_encoders = {}
for column in ['Brand', 'Condition']:
    le = LabelEncoder()
    train_df[column] = le.fit_transform(train_df[column])
    label_encoders[column] = le

# Define Input Features (X) and Target (y)
X = train_df[['Brand', 'RAM', 'Storage', 'Age_Years', 'Condition']]
y = train_df['Resale_Price_INR']

# Split into 80% Training and 20% Testing data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train Random Forest Regressor Model
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Make Predictions on Test data
y_pred = model.predict(X_test)

# Evaluate the Model
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Model Training Complete!")
print(f"Mean Absolute Error (MAE): ₹{mae:.2f}")
print(f"Model Accuracy (R² Score): {r2 * 100:.2f}%")

# Save the Model and Encoders for our Chatbot
model_data = {
    'model': model,
    'label_encoders': label_encoders
}
joblib.dump(model_data, 'price_wise_model.pkl')
print("Model saved successfully as 'price_wise_model.pkl'!")

Model Training Complete!
Mean Absolute Error (MAE): ₹3144.70
Model Accuracy (R² Score): 90.27%
Model saved successfully as 'price_wise_model.pkl'!


### Step 4: Let's Test a Real Prediction!
Now we can write a function to simulate user input from our chatbot and get a price prediction.

In [ ]:
def predict_laptop_price(brand, ram, storage, age, condition):
    # Load model and encoders
    saved_data = joblib.load('price_wise_model.pkl')
    saved_model = saved_data['model']
    encoders = saved_data['label_encoders']

    try:
        # Encode Brand and Condition input using our fitted encoders
        encoded_brand = encoders['Brand'].transform([brand])[0]
        encoded_condition = encoders['Condition'].transform([condition])[0]
    except ValueError as e:
        return f"Error in input values: {e}"

    # Prepare features array
    features = [[encoded_brand, ram, storage, age, encoded_condition]]

    # Predict price
    predicted_val = saved_model.predict(features)[0]

    # Generate a reasonable estimation range
    lower_bound = int(round(predicted_val * 0.9, -2))
    upper_bound = int(round(predicted_val * 1.1, -2))

    return f"💬 PriceWise AI: Mee laptop details batti estimated resale price ₹{lower_bound:,} – ₹{upper_bound:,} undavachu."

# Example test run
sample_prediction = predict_laptop_price(brand='Dell', ram=8, storage=512, age=2.0, condition='Good')
print(sample_prediction)

💬 PriceWise AI: Mee laptop details batti estimated resale price ₹28,800 – ₹35,200 undavachu.


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Step 5: Generate the Streamlit Web Application Code
We will write a python script that outputs `app.py`. This script handles the user interface with interactive forms, sidebar inputs, and displays the price predictions seamlessly using Streamlit.

In [ ]:
app_code = """
import streamlit as st
import joblib
import numpy as np

# Set Page Configuration
st.set_page_config(
    page_title="PriceWise AI — Smart Price Prediction Chatbot",
    page_icon="💻",
    layout="centered"
)

# Custom Header styling
st.title("💻 PriceWise AI")
st.subheader("Smart Used Laptop Price Prediction Chatbot")
st.write("Enter your laptop's specifications below to get an estimated resale price instantly!")

# Load Model and Label Encoders
@st.cache_resource
def load_model_data():
    return joblib.load('price_wise_model.pkl')

try:
    model_data = load_model_data()
    model = model_data['model']
    encoders = model_data['label_encoders']
except Exception as e:
    st.error(f"Error loading the model: {e}. Please make sure 'price_wise_model.pkl' is in the same directory.")
    st.stop()

# Sidebar for helpful info
st.sidebar.header("About PriceWise AI")
st.sidebar.info(
    "This AI Model uses a Random Forest Regressor trained on Indian used laptop market data "
    "to calculate resale values based on brand, age, condition, and key specifications."
)

# Create a Form for User Inputs
with st.form("prediction_form"):
    col1, col2 = st.columns(2)

    with col1:
        brand = st.selectbox("Laptop Brand", options=['Apple', 'Dell', 'HP', 'Lenovo', 'Asus'])
        ram = st.selectbox("RAM Size (GB)", options=[4, 8, 16, 32], index=1)
        storage = st.selectbox("Storage Size (GB)", options=[256, 512, 1024], index=1)

    with col2:
        age = st.slider("Age of Laptop (Years)", min_value=0.5, max_value=6.0, value=2.0, step=0.1)
        condition = st.selectbox("Overall Condition", options=['Excellent', 'Good', 'Fair', 'Poor'], index=1)

    submit_button = st.form_submit_button("Predict Resale Price 🚀")

# Handle prediction on form submit
if submit_button:
    try:
        # Preprocess / Encode the inputs
        encoded_brand = encoders['Brand'].transform([brand])[0]
        encoded_condition = encoders['Condition'].transform([condition])[0]

        # Prepare input features
        features = [[encoded_brand, ram, storage, age, encoded_condition]]

        # Predict value
        prediction = model.predict(features)[0]

        # Generate a reasonable price range (lower and upper bound)
        lower_bound = int(round(prediction * 0.9, -2))
        upper_bound = int(round(prediction * 1.1, -2))

        # Display Result in chatbot-style container
        st.success("🔮 Prediction Ready!")
        st.chat_message("assistant").write(
            f"💬 **PriceWise AI:** Mee laptop details batti estimated resale price "
            f"**Rs. {lower_bound:,} - Rs. {upper_bound:,}** undavachu."
        )
    except Exception as e:
        st.error(f"An error occurred during prediction: {e}")
"""

with open('app.py', 'w') as f:
    f.write(app_code.strip())

print("Streamlit app script successfully saved as 'app.py'!")

Streamlit app script successfully saved as 'app.py'!


### Step 6: How to Publish on GitHub & Deploy Live

Follow these steps to host your PriceWise AI Chatbot online for free:

#### 1. Prepare your files locally
Create a folder on your computer and place these 3 files inside it:
*   `app.py` (The Streamlit UI script you generated)
*   `price_wise_model.pkl` (The trained ML model file)
*   `requirements.txt` (A text file listing the required libraries)

Create a new file named `requirements.txt` in that folder and add these lines:
```text
streamlit
scikit-learn
joblib
pandas
numpy
```

#### 2. Push your project to GitHub
1. Go to [GitHub](https://github.com/) and log in (or sign up if you don't have an account).
2. Click on **New Repository**.
3. Name your repository (e.g., `pricewise-ai-chatbot`) and keep it **Public**.
4. Upload your 3 files (`app.py`, `price_wise_model.pkl`, `requirements.txt`) directly to this repository.

#### 3. Deploy to Streamlit Community Cloud (Free Hosting)
1. Visit [Streamlit Community Cloud](https://share.streamlit.io/) and click **Sign Up** using your GitHub account.
2. Once logged in, click **Create app** (or **New app**).
3. Fill in the deployment details:
   *   **Repository**: Select your GitHub repository (`your-username/pricewise-ai-chatbot`)
   *   **Branch**: Select `main` or `master`
   *   **Main file path**: Enter `app.py`
4. Click **Deploy!**

Within 1-2 minutes, Streamlit will install the requirements and give you a public URL (e.g., `https://pricewise-ai.streamlit.app`) which you can share with your users or judges for the hackathon! 🎉